## 1. 环境与数据读取


In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
from sklearn.datasets import load_iris
from sklearn.decomposition import PCA
from sklearn.feature_selection import (
    VarianceThreshold,
    SelectKBest,
    f_classif,
    chi2,
    mutual_info_classif,
    RFE,
    SelectFromModel,
)
from sklearn.linear_model import Lasso
from sklearn.preprocessing import StandardScaler, MinMaxScaler

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 120)

DATA_PATH = Path("res/AppDataV2.csv")
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

raw_df = pd.read_csv(DATA_PATH, index_col=0)
raw_data = raw_df.drop(columns=["Rating"])
labels = raw_df["Rating"]

print("原始数据形状:", raw_df.shape)
display(raw_df.head())
display(raw_df.dtypes.to_frame("dtype").T)

## 2. 无量纲化


In [ ]:
scaler = StandardScaler()
data = pd.DataFrame(scaler.fit_transform(raw_data), columns=raw_data.columns, index=raw_data.index)

print("特征矩阵:", raw_data.shape, "标签:", labels.shape)
display(data.head())

## 3. Filter：方差选择法


In [ ]:
var_selector = VarianceThreshold(threshold=0.01)
data_after_var_arr = var_selector.fit_transform(raw_data)
var_selected_columns = raw_data.columns[var_selector.get_support()].tolist()

data_after_var = pd.DataFrame(data_after_var_arr, columns=var_selected_columns, index=raw_data.index)
data_after_var_with_label = pd.concat([data_after_var, labels], axis=1)
data_after_var_with_label.to_csv(OUTPUT_DIR / "data_after_var.csv")

print("方差选择后形状:", data_after_var.shape)
print("删除的低方差特征:", raw_data.columns[~var_selector.get_support()].tolist())
display(data_after_var.head())

## 4. Filter：Pearson 相关系数（数值特征）

In [ ]:
numerical_cols = ["Reviews", "Size", "Installs", "Price"]
pearson_rows = []
for col in numerical_cols:
    corr, p_value = pearsonr(raw_data[col], labels)
    pearson_rows.append({"feature": col, "pearson_r": corr, "abs_r": abs(corr), "p_value": p_value})

pearson_summary = pd.DataFrame(pearson_rows).sort_values("abs_r", ascending=False).reset_index(drop=True)
pearson_summary.to_csv(OUTPUT_DIR / "pearson_summary.csv", index=False)

pearson_top_cols = pearson_summary.head(3)["feature"].tolist()
data_numerical = raw_data[pearson_top_cols].copy()

data_numerical.to_csv(OUTPUT_DIR / "filter_pearson_top3.csv")
print("Pearson Top3:", pearson_top_cols)
display(pearson_summary)
display(data_numerical.head())

## 5. Filter：ANOVA（分类/哑变量特征）

In [ ]:
categorical_cols = [
    col for col in raw_data.columns
    if col in ["Type", "Content Rating", "Genres"] or col.startswith("Category_")
]

anova_k = min(30, len(categorical_cols))
anova_selector = SelectKBest(score_func=f_classif, k=anova_k)
data_categorical_arr = anova_selector.fit_transform(raw_data[categorical_cols], labels)
anova_selected_columns = pd.Index(categorical_cols)[anova_selector.get_support()].tolist()

data_categorical = pd.DataFrame(data_categorical_arr, columns=anova_selected_columns, index=raw_data.index)
anova_summary = pd.DataFrame({
    "feature": categorical_cols,
    "f_score": anova_selector.scores_,
    "p_value": anova_selector.pvalues_,
}).sort_values("f_score", ascending=False).reset_index(drop=True)

anova_summary.to_csv(OUTPUT_DIR / "anova_summary.csv", index=False)
data_categorical.to_csv(OUTPUT_DIR / "filter_anova_top30.csv")

print("ANOVA 选择特征数:", len(anova_selected_columns))
display(anova_summary.head(15))
display(data_categorical.head())

## 6. Filter 汇总


In [ ]:
df_after_filter = pd.concat([data_numerical, data_categorical], axis=1)
df_after_filter_with_label = pd.concat([df_after_filter, labels], axis=1)
df_after_filter_with_label.to_csv(OUTPUT_DIR / "df_after_filter.csv")

print("Filter 后形状:", df_after_filter.shape)
display(df_after_filter.head())

## 7. Wrapper：RFE + Lasso


In [ ]:
wrapper_selection = RFE(
    estimator=Lasso(alpha=0.001, max_iter=50000, random_state=42),
    n_features_to_select=30,
    step=1,
)
wrapper_selection.fit(data, labels)
wrapper_selected_columns = raw_data.columns[wrapper_selection.support_].tolist()

df_after_wrapper = raw_data[wrapper_selected_columns].copy()
df_after_wrapper_with_label = pd.concat([df_after_wrapper, labels], axis=1)
df_after_wrapper_with_label.to_csv(OUTPUT_DIR / "df_after_wrapper.csv")

wrapper_ranking = pd.DataFrame({
    "feature": raw_data.columns,
    "selected": wrapper_selection.support_,
    "ranking": wrapper_selection.ranking_,
}).sort_values(["ranking", "feature"]).reset_index(drop=True)
wrapper_ranking.to_csv(OUTPUT_DIR / "wrapper_ranking.csv", index=False)

print("Wrapper 后形状:", df_after_wrapper.shape)
display(wrapper_ranking.head(40))
display(df_after_wrapper.head())

## 8. Embedded：SelectFromModel + Lasso


In [ ]:
lasso = Lasso(alpha=0.001, max_iter=50000, random_state=42)
lasso.fit(data, labels)
embedded_model = SelectFromModel(lasso, prefit=True, threshold="mean")
embedded_support = embedded_model.get_support()
embedded_selected_columns = raw_data.columns[embedded_support].tolist()

df_after_embedded = raw_data[embedded_selected_columns].copy()
df_after_embedded_with_label = pd.concat([df_after_embedded, labels], axis=1)
df_after_embedded_with_label.to_csv(OUTPUT_DIR / "df_after_embedded.csv")

embedded_importance = pd.DataFrame({
    "feature": raw_data.columns,
    "coef": lasso.coef_,
    "abs_coef": np.abs(lasso.coef_),
    "selected": embedded_support,
}).sort_values("abs_coef", ascending=False).reset_index(drop=True)
embedded_importance.to_csv(OUTPUT_DIR / "embedded_lasso_coef.csv", index=False)

print("Embedded 后形状:", df_after_embedded.shape)
display(embedded_importance.head(20))
display(df_after_embedded.head())

## 9. PCA 降维


In [ ]:
pca = PCA(n_components=0.85, random_state=42)
data_after_pca = pd.DataFrame(
    pca.fit_transform(data),
    columns=[f"PC{i + 1}" for i in range(pca.n_components_)],
    index=raw_data.index,
)
data_after_pca_withlabels = pd.concat([data_after_pca, labels], axis=1)
data_after_pca_withlabels.to_csv(OUTPUT_DIR / "after_pca.csv")

explained_summary = pd.DataFrame({
    "component": data_after_pca.columns,
    "explained_variance_ratio": pca.explained_variance_ratio_,
    "cumulative_ratio": np.cumsum(pca.explained_variance_ratio_),
})
explained_summary.to_csv(OUTPUT_DIR / "pca_explained_variance.csv", index=False)

print(f"PCA 后维度: {raw_data.shape[1]} -> {pca.n_components_}")
print(f"累计解释方差: {pca.explained_variance_ratio_.sum():.4f}")
display(explained_summary.head(15))
display(data_after_pca.head())

## 10. PCA 可视化


In [ ]:
plot_df = data_after_pca_withlabels.copy()
plot_df["RatingLevel"] = plot_df["Rating"].round().astype(int).clip(1, 5)

plt.figure(figsize=(8, 6))
for rating_level in sorted(plot_df["RatingLevel"].unique()):
    subset = plot_df[plot_df["RatingLevel"] == rating_level]
    plt.scatter(subset["PC1"], subset["PC2"], s=12, alpha=0.55, label=str(rating_level))

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("PCA Result by Rounded Rating")
plt.legend(title="Rating")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "pca_scatter.png", dpi=160)
plt.show()

## 11. 思考题：Iris 数据集 Filter 方法

In [ ]:
iris = load_iris()
X = pd.DataFrame(iris.data, columns=iris.feature_names)
y = pd.Series(iris.target, name="target")
iris_df = pd.concat([X, y], axis=1)

print("Iris 数据形状:", iris_df.shape)
display(iris_df.sample(5, random_state=42))

### 11.1 方差选择法


In [ ]:
iris_var_selector = VarianceThreshold(threshold=0.6)
iris_after_var = pd.DataFrame(
    iris_var_selector.fit_transform(X),
    columns=X.columns[iris_var_selector.get_support()],
)
iris_var_summary = pd.DataFrame({
    "feature": X.columns,
    "variance": X.var(ddof=0).values,
    "selected": iris_var_selector.get_support(),
}).sort_values("variance", ascending=False).reset_index(drop=True)

iris_after_var.to_csv(OUTPUT_DIR / "iris_after_variance.csv", index=False)
iris_var_summary.to_csv(OUTPUT_DIR / "iris_variance_summary.csv", index=False)

display(iris_var_summary)
display(iris_after_var.head())

### 11.2 相关系数法


In [ ]:
iris_pearson_rows = []
for col in X.columns:
    corr, p_value = pearsonr(X[col], y)
    iris_pearson_rows.append({"feature": col, "pearson_r": corr, "abs_r": abs(corr), "p_value": p_value})

iris_pearson_summary = pd.DataFrame(iris_pearson_rows).sort_values("abs_r", ascending=False).reset_index(drop=True)
iris_pearson_top2 = iris_pearson_summary.head(2)["feature"].tolist()
iris_after_pearson = X[iris_pearson_top2].copy()

iris_pearson_summary.to_csv(OUTPUT_DIR / "iris_pearson_summary.csv", index=False)
iris_after_pearson.to_csv(OUTPUT_DIR / "iris_after_pearson_top2.csv", index=False)

print("Pearson Top2:", iris_pearson_top2)
display(iris_pearson_summary)
display(iris_after_pearson.head())

### 11.3 卡方检验法


In [ ]:
iris_chi2_selector = SelectKBest(score_func=chi2, k=2)
iris_after_chi2 = pd.DataFrame(
    iris_chi2_selector.fit_transform(X, y),
    columns=X.columns[iris_chi2_selector.get_support()],
)
iris_chi2_summary = pd.DataFrame({
    "feature": X.columns,
    "chi2_score": iris_chi2_selector.scores_,
    "p_value": iris_chi2_selector.pvalues_,
    "selected": iris_chi2_selector.get_support(),
}).sort_values("chi2_score", ascending=False).reset_index(drop=True)

iris_after_chi2.to_csv(OUTPUT_DIR / "iris_after_chi2_top2.csv", index=False)
iris_chi2_summary.to_csv(OUTPUT_DIR / "iris_chi2_summary.csv", index=False)

display(iris_chi2_summary)
display(iris_after_chi2.head())

### 11.4 互信息法


In [ ]:
mi_scores = mutual_info_classif(X, y, random_state=42)
iris_mi_summary = pd.DataFrame({
    "feature": X.columns,
    "mutual_info": mi_scores,
}).sort_values("mutual_info", ascending=False).reset_index(drop=True)
iris_mi_top2 = iris_mi_summary.head(2)["feature"].tolist()
iris_after_mi = X[iris_mi_top2].copy()

iris_mi_summary.to_csv(OUTPUT_DIR / "iris_mutual_info_summary.csv", index=False)
iris_after_mi.to_csv(OUTPUT_DIR / "iris_after_mutual_info_top2.csv", index=False)

print("互信息 Top2:", iris_mi_top2)
display(iris_mi_summary)
display(iris_after_mi.head())